In [1]:
# ==========================================
# TITANIC SURVIVAL PREDICTION
# END TO END MACHINE LEARNING PIPELINE
# ==========================================

# ======================
# IMPORT LIBRARIES
# ======================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [3]:
# ======================
# LOAD DATA
# ======================

train_df = pd.read_csv("./Data/train.csv")
test_df = pd.read_csv("./Data/test.csv")

# Save PassengerId for submission file
test_passenger_id = test_df["PassengerId"]

# ======================
# PREPROCESSING FUNCTION
# ======================

def preprocess_data(df):

    # Make copy
    df = df.copy()

    # ----------------------
    # DROP UNNECESSARY COLUMNS
    # ----------------------

    drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]

    existing_cols = [col for col in drop_cols if col in df.columns]

    df.drop(columns=existing_cols, inplace=True)

    # ----------------------
    # HANDLE MISSING VALUES
    # ----------------------

    # Age -> median
    if "Age" in df.columns:
        df["Age"] = df["Age"].fillna(df["Age"].median())

    # Fare -> median
    if "Fare" in df.columns:
        df["Fare"] = df["Fare"].fillna(df["Fare"].median())

    # Embarked -> mode
    if "Embarked" in df.columns:
        df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

    # ----------------------
    # LABEL ENCODING
    # ----------------------

    if "Sex" in df.columns:
        df["Sex"] = df["Sex"].map({
            "male": 0,
            "female": 1
        })

    # ----------------------
    # ONE HOT ENCODING
    # ----------------------

    if "Embarked" in df.columns:
        df = pd.get_dummies(
            df,
            columns=["Embarked"],
            drop_first=True
        )

    return df

In [4]:
# ======================
# APPLY PREPROCESSING
# ======================

train_df = preprocess_data(train_df)
test_df = preprocess_data(test_df)

# ======================
# FEATURES AND TARGET
# ======================

X = train_df.drop("Survived", axis=1)

y = train_df["Survived"]

# ======================
# TRAIN TEST SPLIT
# ======================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ======================
# FEATURE SCALING
# ======================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_valid_scaled = scaler.transform(X_valid)

test_scaled = scaler.transform(test_df)

# ======================
# MODEL TRAINING
# ======================

model = LogisticRegression()

model.fit(X_train_scaled, y_train)

# ======================
# VALIDATION PREDICTION
# ======================

y_pred = model.predict(X_valid_scaled)

# ======================
# EVALUATION
# ======================

accuracy = accuracy_score(y_valid, y_pred)

print("Accuracy Score:")
print(accuracy)

print("\n==========================")
print("Confusion Matrix")
print("==========================")

print(confusion_matrix(y_valid, y_pred))

print("\n==========================")
print("Classification Report")
print("==========================")

print(classification_report(y_valid, y_pred))

# ======================
# TEST DATA PREDICTION
# ======================

test_predictions = model.predict(test_scaled)

# ======================
# CREATE SUBMISSION FILE
# ======================

submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Survived": test_predictions
})

submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv file created successfully!")

Accuracy Score:
0.8100558659217877

Confusion Matrix
[[90 15]
 [19 55]]

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


submission.csv file created successfully!
